# Live llm-d comparison: round-robin and prefix-aware scheduling

**Run All** requests a fresh, bounded experiment on the two existing Aurora Qwen backends. The notebook shows only the result associated with this run's new request ID. An unavailable controller produces an explicit error; it never substitutes a saved measurement.

A presenter-operated administrator controller performs the comparison through temporary loopback port forwards. The Workbench receives no administrator credential or direct backend access. This is a supervised showroom exercise, not a permanently available benchmarking service.

## 1. Check the supervised session

Select **Aurora Inference Demo** as the kernel. The presenter must first start the short-lived controller. Its lease allows at most six comparisons, one at a time, with a 30-second cooldown.

Each comparison runs two fixed phases: **round-robin** and **llm-d**, each limited to 30 seconds, 16 requests, concurrency 2, and 32 output tokens. One additional authorized 16-token preflight and one denied anonymous check verify the native authentication boundary. No model, replica, or routing configuration is changed.

In [ ]:
from llmd_workbench import controller_status, compare, display_comparison

controller_status()

## 2. Run one fresh comparison

The first phase explicitly alternates requests between the two backends through administrator-only local forwards. The second uses the authenticated llm-d Gateway and its installed endpoint picker, including the prefix-cache scorer.

The phases use the same workload shape, but distinct fresh prefix salts. Prompt bytes are therefore different. Existing caches are not cleared. Other users may be active.

This cell waits for at most **210 seconds**. **Interrupt Kernel** writes a cancellation request; the controller detects it and stops the active process group. Detection may take several seconds, and an already accepted server request may finish. Closing the browser alone does not send cancellation.

In [ ]:
live_comparison = await compare()

## 3. Inspect the measurements from this run

The tables show completed requests, failures, observed per-backend completion counters, endpoint-picker activity, local prefix-cache counters, and GuideLLM latency/throughput. Charts use only the returned measurements. A phase with errors or incomplete requests fails the completion gate.

**TTFT** is GuideLLM's reported client streaming time to first token. **End-to-end latency** is measured in seconds in the table and milliseconds in the chart. Short samples do not support reliable tail-latency or capacity claims.

In [ ]:
display_comparison(live_comparison)

## 4. Explain what the customer can observe

- **Round-robin** sends alternating requests to the two available backends. Their shared counters can also include unrelated requests.
- **llm-d** chooses an endpoint using the installed scoring configuration. A prefix-cache scorer is present alongside queue, cache-utilization, and other scoring components; this exercise does not isolate any one scorer's effect.
- **Local prefix-cache hits** show reuse observed within backend caches. They do not demonstrate cross-node KV transfer or prefill/decode disaggregation.
- **EPP activity** and backend deltas are shared counters, not a per-request trace. A larger delta cannot be attributed exclusively to this notebook.
- **Latency differences** include different authentication and transport paths, cache state, and concurrent activity. A faster value in either phase is an observation from this small run, not a causal routing-speedup claim.

For native charts, open **Observe & monitor → Dashboard** and recheck **Project = ai-showroom**, **Model = aurora-qwen-4b** on each relevant tab. Use **LLM Traffic**, **LLM Utilization**, and **LLM Performance** to inspect the same UTC window. Scrape intervals may miss a short burst; native graphs include all callers. Native inter-token latency is currently unavailable for this runtime's metric mapping—a displayed fallback zero is not a measurement.

The private Qwen path is separate from the MaaS load subscription shown in **Usage**. See the [native dashboard guide](https://weslleyrosalem.com/rhoai-showroom/operations/native-dashboards/) and [AHEAD llm-d lab](https://weslleyrosalem.com/rhoai-showroom/labs/ahead-llmd/) for query and methodology limits.

The typed response remains on the Workbench's persistent storage under the request ID. Saving this notebook preserves the currently displayed result; **Run All always requests another fresh comparison** while the supervised session has capacity.